In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2024_Ashok_Vihar_Delhi_DPCC_2024.xlsx")

In [3]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,354.0,NaN,185.0,144.0,215.0,257.0,89.0,49.0,86.0,120.0,373.0,298.0
1,2,345.0,227.0,108.0,148.0,186.0,144.0,91.0,59.0,88.0,144.0,351.0,291.0
2,3,317.0,218.0,128.0,172.0,267.0,130.0,94.0,57.0,92.0,136.0,421.0,289.0
3,4,372.0,305.0,135.0,181.0,271.0,178.0,51.0,50.0,65.0,151.0,420.0,170.0
4,5,329.0,221.0,132.0,184.0,305.0,208.0,75.0,44.0,162.0,127.0,401.0,151.0
5,6,322.0,147.0,116.0,177.0,260.0,174.0,49.0,42.0,84.0,121.0,397.0,189.0
6,7,342.0,177.0,177.0,167.0,323.0,188.0,42.0,40.0,71.0,110.0,430.0,246.0
7,8,358.0,164.0,144.0,189.0,243.0,196.0,46.0,41.0,86.0,140.0,406.0,334.0
8,9,326.0,138.0,131.0,190.0,174.0,155.0,70.0,49.0,101.0,136.0,372.0,223.0
9,10,257.0,322.0,149.0,192.0,176.0,149.0,162.0,54.0,91.0,111.0,355.0,268.0


In [4]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [5]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [6]:
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [7]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,354.000000,192.909091,185.000000,144.000000,215.0,140.294118,89.000000,49.000000,86.000000,120.000000,373.000000,298.0
1,2,345.000000,227.000000,108.000000,148.000000,186.0,144.000000,91.000000,59.000000,88.000000,144.000000,351.000000,291.0
2,3,317.000000,218.000000,128.000000,172.000000,267.0,130.000000,94.000000,57.000000,92.000000,136.000000,421.000000,289.0
3,4,372.000000,305.000000,135.000000,181.000000,271.0,178.000000,51.000000,50.000000,65.000000,151.000000,420.000000,170.0
4,5,329.000000,221.000000,132.000000,184.000000,305.0,208.000000,75.000000,44.000000,90.828571,127.000000,401.000000,151.0
5,6,322.000000,147.000000,116.000000,177.000000,260.0,174.000000,49.000000,42.000000,84.000000,121.000000,397.000000,189.0
6,7,342.000000,177.000000,177.000000,167.000000,323.0,188.000000,42.000000,40.000000,71.000000,110.000000,430.000000,246.0
7,8,358.000000,164.000000,144.000000,189.000000,243.0,196.000000,46.000000,41.000000,86.000000,140.000000,406.000000,334.0
8,9,326.000000,138.000000,131.000000,190.000000,174.0,155.000000,70.000000,49.000000,101.000000,136.000000,372.000000,223.0
9,10,257.000000,322.000000,149.000000,192.000000,176.0,149.000000,78.828571,54.000000,91.000000,111.000000,355.000000,268.0
